In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
from numba import njit
from pynamicalsys import DiscreteDynamicalSystem as dds, PlotStyler
from string import ascii_lowercase as abc
import os

if not os.path.isdir("figures"):
    os.mkdir("figures")

ds = dds(model="extended standard nontwist map")
a = 0.53 
b = 0.53
m = 0.8

# Fig. 1

In [ ]:
c = 5e-3
ms = [0.0, 0.4, 0.8, 1.0, (np.sqrt(5) + 1) / 2, 1.8, 2.0, 2.3, 3.0]
num_ic = 250
total_time = 20000

In [ ]:
trajectories = []
for m in ms:
    parameters = [a, b, c, m]
    np.random.seed(1312)
    x = np.random.uniform(0.3, 0.7, num_ic)
    y = np.random.uniform(-0.6, 0.6, num_ic)
    u = np.stack([x, y]).T
    trajectories.append(ds.trajectory(u, total_time, parameters))

In [ ]:
fontsize = 18
ps = PlotStyler(fontsize=fontsize, markersize=0.1, markeredgewidth=0)
ps.apply_style()

fig, ax = plt.subplots(3, 3, sharex=True, sharey=True, figsize=(10, 7))

for i in range(len(trajectories)):
    ps.set_tick_padding(ax.flat[i], pad_x=6)
    # ax.flat[i].plot(trajectories[i][:, 0], trajectories[i][:, 1], "ko")
    ax.flat[i].scatter(trajectories[i][:, 0], trajectories[i][:, 1], color="k", s=0.01, edgecolor="none")
plt.xlim(0, 1)
plt.ylim(-0.5, 0.5)

for i in range(3):
    ax[-1, i].set_xlabel("$x$")
    ax[i, 0].set_ylabel("$y$")

xbox = 0.01
ybox = 0.901
bbox = {"facecolor": "w", "linewidth": 0.0, "pad": 1, "alpha": 0.75}
for i in range(9):
    ax.flat[i].text(xbox, ybox, f"({abc[i]})", transform=ax.flat[i].transAxes, bbox=bbox)

plt.subplots_adjust(left=0.079, bottom=0.072, right=0.982, top=0.989, hspace=0.1, wspace=0.15)
plt.savefig(f"figures/fig1.png", dpi=400)
plt.close()

# Fig. 2

In [ ]:
m = 0.8
c_vals = [0, 1e-5, 5e-5, 1e-4, 5e-4, 1e-3]
num_ic = 250
total_time = 10000
y_ranges = [(-0.1, 0.5), (-0.45, 0.05)]

In [ ]:
fontsize = 18
ps = PlotStyler(fontsize=fontsize, markersize=0.1, markeredgewidth=0)
ps.apply_style()

# fig, ax = plt.subplots(2, len(c_vals), sharex=True, figsize=(5, 7))
fig, ax = plt.subplots(2, len(c_vals), sharex=True, figsize=(10, 4))
s = 0.01
np.random.seed(13)
for i in range(2):
    for j, c in enumerate(c_vals):
        parameters = [a, b, c, m]
        x = np.random.uniform(0.38, 0.62, num_ic)
        y = np.random.uniform(*y_ranges[i], num_ic)
        u = np.stack([x, y]).T
        trajectories = ds.trajectory(u, total_time, parameters)

        ax[i, j].scatter(trajectories[:, 0], trajectories[:, 1], color="k", s=s, edgecolor="none")
        # ax[i, j].plot(trajectories[:, 0], trajectories[:, 1], "ko")
        # ax[i].plot(trajectories[:, 0], trajectories[:, 1], "ko")
        ax[i, j].set_ylim(*y_ranges[i])
        if j > 0:
            ax[i, j].set_yticklabels([])


xbox = 0.023
ybox = 0.88
bbox = {"facecolor": "w", "linewidth": 0.0, "pad": 1, "alpha": 0.75}
for i in range(2*len(c_vals)):
    ii = i % 2
    jj = i // 2
    ax[ii, jj].text(xbox, ybox, f"({abc[i]})", transform=ax[ii, jj].transAxes, bbox=bbox)
for i in range(len(c_vals)):    
    ax[-1, i].set_xlabel("$x$")
for i in range(2):
    ax[i, 0].set_ylabel("$y$")

    
plt.xlim(0.38, 0.62)
plt.subplots_adjust(left=0.07, bottom=0.12, right=0.995, top=0.99, hspace=0.05, wspace=0.05)
plt.savefig("figures/fig2.png", dpi=400)
plt.close()

# Fig. 3

In [ ]:
x_range = (0.38, 0.62)
y_ranges = [(-0.1, 0.5), (-0.45, 0.05)]

c_vals = [5e-5, 1e-4, 5e-4, 1e-3]
a = 0.53
m = 0.80
total_time = int(1e4)
total_time_sali = int(1e4)
grid_size = 500


In [ ]:
ps = PlotStyler(fontsize=18)
ps.apply_style()

fig, ax = plt.subplots(4, len(c_vals), sharex=True, figsize=(10, 7))

for j, c in enumerate(c_vals):
    for i, y_range in enumerate(y_ranges):
        datafile = f"data/grid_escape_times_a={a:.5f}_b={b:.5f}_c={c:.5f}_m={m:.5f}_xrange={x_range[0]:.5f}_{x_range[1]:.5f}_yrange={y_range[0]:.5f}_{y_range[1]:.5f}_time={total_time}_grid_size={grid_size}.dat"
        df = pd.read_csv(datafile, header=None, sep=r"\s+")
        x = np.array(df[0]).reshape(grid_size, grid_size)
        y = np.array(df[1]).reshape(grid_size, grid_size)
        z = abs(np.array(df[2]).reshape(grid_size, grid_size))
        z[z < 1e-16] = 1e-16
        hm = ax[i, j].pcolormesh(x, y, z, cmap="nipy_spectral", norm=mpl.colors.LogNorm(vmin=1e0, vmax=total_time))
        ax[0, j].set_yticks([0, 0.2, 0.4])
        ax[1, j].set_yticks([-0.4, -0.2, 0])
        if j > 0:
            ax[0, j].set_yticklabels([])
            ax[1, j].set_yticklabels([])
ax[0, 0].set_xticks([0.4, 0.5, 0.6])
ax[0, 0].set_ylabel("$y$")
ax[1, 0].set_ylabel("$y$")
ax[-1, 0].set_xlabel("$x$")
ax[-1, 1].set_xlabel("$x$")
ax[-1, 2].set_xlabel("$x$")
ax[-1, 3].set_xlabel("$x$")

for j, c in enumerate(c_vals):
    for i, y_range in enumerate(y_ranges):
        datafile = f"data/grid_sali_a={a:.5f}_b={b:.5f}_c={c:.5f}_m={m:.5f}_xrange={x_range[0]:.5f}_{x_range[1]:.5f}_yrange={y_range[0]:.5f}_{y_range[1]:.5f}_time={total_time_sali}_grid_size={grid_size}.dat"
        df = pd.read_csv(datafile, header=None, sep=r"\s+")
        x = np.array(df[0]).reshape(grid_size, grid_size)
        y = np.array(df[1]).reshape(grid_size, grid_size)
        z = abs(np.array(df[2]).reshape(grid_size, grid_size))
        z[z < 1e-16] = 1e-16
        hm2 = ax[2 + i, j].pcolormesh(x, y, z, cmap="nipy_spectral", norm=mpl.colors.LogNorm(vmin=1e-16, vmax=np.sqrt(2)))
        ax[2, j].set_yticks([0, 0.2, 0.4])
        ax[3, j].set_yticks([-0.4, -0.2, 0])
        if j > 0:
            ax[2, j].set_yticklabels([])
            ax[3, j].set_yticklabels([])
ax[2, 0].set_ylabel("$y$")
ax[3, 0].set_ylabel("$y$")

xbox = 0.015
ybox = 0.868
bbox = {"facecolor": "w", "linewidth": 0.0, "pad": 1, "alpha": 0.75}
for i in range(len(ax.flat)):
    ax.flat[i].text(xbox, ybox, f"({abc[i]})", transform=ax.flat[i].transAxes, bbox=bbox)
plt.subplots_adjust(left=0.0682, bottom=0.07, right=1.065, top=0.98, wspace=0.05, hspace=0.05)

plt.colorbar(hm, ax=ax[:2], label="Escape time", aspect=40, pad=0.01)
plt.colorbar(hm2, ax=ax[2:], label="SALI", aspect=40, pad=0.01)

plt.savefig("figures/fig3.png", dpi=400)
plt.close()

# Fig. 4

In [ ]:
m = 0.8
c_range = [1e-16, 0.5]
total_time = int(1e4)
y_ranges = [(0.0, 0.4), (-0.45, 0.0)]
grid_size = 500

In [ ]:
ps = PlotStyler(fontsize=18)
ps.apply_style()

fig, ax = plt.subplots(1, 2, sharex=True, figsize=(10, 3.5))
ps.set_tick_padding(ax[0], pad_x=6)
ps.set_tick_padding(ax[1], pad_x=6)
for i, y_range in enumerate(y_ranges):
    datafile = f"data/cgbd_sali_a={a:.5f}_b={b:.5f}_m={m:.5f}_crange={c_range[0]:.16f}_{c_range[1]:.16f}_yrange={y_range[0]:.5f}_{y_range[1]:.5f}_time={total_time}_grid_size={grid_size}.dat"
    df = pd.read_csv(datafile, header=None, sep=r"\s+")
    c = np.array(df[0]).reshape(grid_size, grid_size)
    y = np.array(df[1]).reshape(grid_size, grid_size)
    sali = np.array(df[2]).reshape(grid_size, grid_size)
    sali[sali < 1e-16] = 1e-16
    hm = ax[i].pcolormesh(c, y, sali, cmap="nipy_spectral", norm=mpl.colors.LogNorm(vmin=1e-16, vmax=np.sqrt(2)))
ax[0].set_xscale("log")
ax[0].set_xlim(c.min(), 0.1)
ax[0].set_ylabel("$y_0$")
ax[0].set_xlabel("$c$")
ax[1].set_xlabel("$c$")
plt.subplots_adjust(left=0.055, bottom=0.15, right=1.07, top=0.98, wspace=0.15)
plt.colorbar(hm, ax=ax, label="SALI", pad=0.008, aspect=40)
xbox = 0.01
ybox = 0.926
bbox = {"facecolor": "w", "linewidth": 0.0, "pad": 1, "alpha": 0.75}
for i in range(len(ax.flat)):
    ax.flat[i].text(xbox, ybox, f"({abc[i]})", transform=ax.flat[i].transAxes, bbox=bbox)
plt.savefig("figures/fig4.png", dpi=400)

# Fig. 5

In [ ]:
a = 0.53 
b = 0.53
m = 0.8
c_list = [0, 1e-5, 5e-5, 1e-4, 5e-4, 1e-3]
total_time = int(1e8)
colors = sns.color_palette("hls", len(c_list))

In [ ]:
ps = PlotStyler(fontsize=18, legend_fontsize=12, linewidth=0.5)
ps.apply_style()

fig, ax = plt.subplots(1, 2, sharex=True, sharey=True, figsize=(10, 3))
ps.set_tick_padding(ax[0], pad_x=6)
ps.set_tick_padding(ax[1], pad_x=6)

for j, ul in enumerate(["U", "L"]):
    for i, c in enumerate(c_list):
        try:
            df = f"data/mle_history_{ul}_a={a:.5f}_b={b:.5f}_c={c:.5f}_m={m:.5f}_time={total_time}.dat"
            df = pd.read_csv(df, header=None, sep=r"\s+")
            x = np.array(df[0])
            y = np.array(df[1])
            # ax[0, j].loglog(x[:-1], y[:-1], c=colors[i], label=f"$c = {c:.5f}$")
            ax[j].loglog(x[:-1], y[:-1], c=colors[i])
            ax[j].loglog(0, 0, c=colors[i], label=f"$c = {c:.5f}$", lw=1)
        except:
            pass
    ax[j].set_axisbelow(False)

ax[0].set_ylabel(r"$\lambda_1$")
ax[0].set_xlim(1, 1e12)
ax[0].set_ylim(1e-13, 5e1)
ax[0].set_xlabel("$n$")
ax[1].set_xlabel("$n$")
x_new = np.logspace(0, 12, 1000)
y_new = 0.3 * x_new ** -1
ax[0].plot(x_new, y_new, "k--", lw=1)
ax[1].plot(x_new, y_new, "k--", lw=1)
handles, labels = ax[0].get_legend_handles_labels()

fig.legend(
    handles,
    labels,
    loc="upper center",
    ncol=len(labels),
    frameon=False,
    columnspacing=0.75,
    handletextpad=0.4,
    bbox_to_anchor=(0.525, 1.02),
)

xbox = 0.007
ybox = 0.907
bbox = {"facecolor": "w", "linewidth": 0.0, "pad": 1, "alpha": 0.75}
for i in range(2):
    ax[i].text(xbox, ybox, f"({abc[i]})", transform=ax[i].transAxes, bbox=bbox)
plt.subplots_adjust(left=0.08, bottom=0.17, right=0.981, top=0.9, wspace=0.08,
                    hspace=0.1)

plt.savefig("figures/fig5.png", dpi=400)

# Fig. 6

In [ ]:
m = 0.8
c = 1e-4
parameters = [a, b, c, m]
ds.set_parameters(parameters)
num_ic = 100
total_time = 20000

In [ ]:
trajectories = []
y_ranges = [(0.2, 0.28), (-0.28, -0.18)]
x_ranges = (0.48, 0.52)
np.random.seed(1213)
for y_range in y_ranges:
    # x = np.random.uniform(*x_range, num_ic)
    x = np.ones(num_ic) * 0.5
    y = np.random.uniform(*y_range, num_ic)
    y = np.linspace(*y_range, num_ic)
    u = np.stack([x, y]).T
    traj = ds.trajectory(u, total_time, parameters)
    trajectories.append(traj)

In [ ]:
trajectories_zoom = []
y_ranges_zoom = [(0.23, 0.246), (-0.25, -0.225)]
x_ranges_zoom = [(0.495, 0.505), (0.495, 0.505)]
np.random.seed(1213)
for y_range in y_ranges_zoom:
    # x = np.random.uniform(*x_range_zoom, num_ic)
    x = np.ones(num_ic) * 0.5
    y = np.random.uniform(*y_range, num_ic)
    y = np.linspace(*y_range, num_ic)
    u = np.stack([x, y]).T
    traj = ds.trajectory(u, total_time, parameters)
    trajectories_zoom.append(traj)

In [ ]:
trajectories_zoom2 = []
x_ranges_zoom2 = [(0.499, 0.501), (0.499, 0.501)]
y_ranges_zoom2 = [(0.236, 0.24), (-0.242, -0.234)]
np.random.seed(1213)
for y_range in y_ranges_zoom2:
    # x = np.random.uniform(*x_range_zoom, num_ic)
    x = np.ones(num_ic) * 0.5
    y = np.random.uniform(*y_range, num_ic)
    y = np.linspace(*y_range, num_ic)
    u = np.stack([x, y]).T
    traj = ds.trajectory(u, total_time, parameters)
    trajectories_zoom2.append(traj)

In [ ]:
ps = PlotStyler(fontsize=18, legend_fontsize=12, linewidth=0.5)
ps.apply_style()

fig, ax = plt.subplots(2, 3, figsize=(10, 4))

for i in range(6):
    ps.set_tick_padding(ax.flat[i], pad_x=7)

s = .05
ax[0, 0].scatter(trajectories[0][:, 0], trajectories[0][:, 1], color="k", s=s, edgecolor="none")
ax[0, 1].scatter(trajectories_zoom[0][:, 0], trajectories_zoom[0][:, 1], color="k", s=s, edgecolor="none")
ax[0, 2].scatter(trajectories_zoom2[0][:, 0], trajectories_zoom2[0][:, 1], color="k", s=s, edgecolor="none")

ax[1, 0].scatter(trajectories[1][:, 0], trajectories[1][:, 1], color="k", s=s, edgecolor="none")
ax[1, 1].scatter(trajectories_zoom[1][:, 0], trajectories_zoom[1][:, 1], color="k", s=s, edgecolor="none")
ax[1, 2].scatter(trajectories_zoom2[1][:, 0], trajectories_zoom2[1][:, 1], color="k", s=s, edgecolor="none")

ax[0, 0].set_xlim(*x_ranges)
ax[0, 0].set_ylim(*y_ranges[0])
ax[0, 1].set_xlim(*x_ranges_zoom[0])
ax[0, 1].set_ylim(*y_ranges_zoom[0])
ax[0, 2].set_xlim(*x_ranges_zoom2[0])
ax[0, 2].set_ylim(*y_ranges_zoom2[0])

ax[1, 0].set_xlim(*x_ranges)
ax[1, 0].set_ylim(*y_ranges[1])
ax[1, 1].set_xlim(*x_ranges_zoom[1])
ax[1, 1].set_ylim(*y_ranges_zoom[1])
ax[1, 2].set_xlim(*x_ranges_zoom2[1])
ax[1, 2].set_ylim(*y_ranges_zoom2[1])


ax[1, 0].set_ylim(*y_ranges[1])

for i in range(2):
    xs = [x_ranges_zoom[i][0], x_ranges_zoom[i][1], x_ranges_zoom[i][1], x_ranges_zoom[i][0], x_ranges_zoom[i][0]]
    ys = [y_ranges_zoom[i][0], y_ranges_zoom[i][0], y_ranges_zoom[i][1], y_ranges_zoom[i][1], y_ranges_zoom[i][0]]

    ax[i, 0].plot(xs, ys, "r--", linewidth=1)

    xs = [x_ranges_zoom2[i][0], x_ranges_zoom2[i][1], x_ranges_zoom2[i][1], x_ranges_zoom2[i][0], x_ranges_zoom2[i][0]]
    ys = [y_ranges_zoom2[i][0], y_ranges_zoom2[i][0], y_ranges_zoom2[i][1], y_ranges_zoom2[i][1], y_ranges_zoom2[i][0]]

    ax[i, 1].plot(xs, ys, "r--", linewidth=1)

from matplotlib.patches import ConnectionPatch

for i in range(2):
    # ---- zoom (col 0 -> col 1) ----
    x0, x1 = x_ranges_zoom[i]
    y0, y1 = y_ranges_zoom[i]

    # connect right edge of rectangle to left edge of next panel
    con1 = ConnectionPatch(
        xyA=(x1, y0), coordsA=ax[i, 0].transData,
        xyB=(x_ranges_zoom[i][0], y_ranges_zoom[i][0]), coordsB=ax[i, 1].transData,
        color="r", linestyle="--", linewidth=1, alpha=0.5
    )

    con2 = ConnectionPatch(
        xyA=(x1, y1), coordsA=ax[i, 0].transData,
        xyB=(x_ranges_zoom[i][0], y_ranges_zoom[i][1]), coordsB=ax[i, 1].transData,
        color="r", linestyle="--", linewidth=1, alpha=0.5
    )

    fig.add_artist(con1)
    fig.add_artist(con2)

    # ---- zoom2 (col 1 -> col 2) ----
    x0, x1 = x_ranges_zoom2[i]
    y0, y1 = y_ranges_zoom2[i]

    con3 = ConnectionPatch(
        xyA=(x1, y0), coordsA=ax[i, 1].transData,
        xyB=(x_ranges_zoom2[i][0], y_ranges_zoom2[i][0]), coordsB=ax[i, 2].transData,
        color="r", linestyle="--", linewidth=1, alpha=0.5
    )
    con4 = ConnectionPatch(
        xyA=(x1, y1), coordsA=ax[i, 1].transData,
        xyB=(x_ranges_zoom2[i][0], y_ranges_zoom2[i][1]), coordsB=ax[i, 2].transData,
        color="r", linestyle="--", linewidth=1, alpha=0.5
    )

    fig.add_artist(con3)
    fig.add_artist(con4)
  
for axis in ax.flat:
    axis.set_axisbelow(False)

ax[0, 0].set_ylabel("$y$")
ax[1, 0].set_ylabel("$y$")
ax[-1, 0].set_xlabel("$x$")
ax[-1, 1].set_xlabel("$x$")
ax[-1, 2].set_xlabel("$x$")

xbox = 0.011
ybox = 0.871
bbox = {"facecolor": "w", "linewidth": 0.0, "pad": 1, "alpha": 0.75}
for i in range(6):
    ax.flat[i].text(xbox, ybox, f"({abc[i]})", transform=ax.flat[i].transAxes, bbox=bbox)


plt.subplots_adjust(left=0.0785, bottom=0.13, right=0.98, top=0.98, hspace=0.2, wspace=0.25)
plt.savefig("figures/fig6.png", dpi=400)
plt.close()

# Fig. 7

In [ ]:
a = 0.53 
b = 0.53
m = 0.8
c = 1e-4
total_time = int(5e6)
finite_time = 200
rr = 0.05
rte_lims = {"U": (0, 1.6),
            "L": (0., 1.6)}#rte.min(), rte.max()

In [ ]:
xs = {}
ys = {}
rtes = {}
probs = {}
bin_centers = {}
bar_colors = {}
bin_edges = {}
norms = {}
for ul in ["U", "L"]:
    df = f"data/finite_time_rte_{ul}_a={a:.5f}_b={b:.5f}_c={c:.5f}_m={m:.5f}_time={total_time}_ftime={finite_time}_rr={rr:.3f}.dat"
    df = pd.read_csv(df, header=None, sep=r"\s+")
    X = np.array(df[0])
    Y = np.array(df[1])
    rte = np.array(df[2])

    mask = rte > 0
    x = X[mask]
    y = Y[mask]
    rte = rte[mask]
    xs[ul] = x
    ys[ul] = y
    rtes[ul] = rte

    cmap = plt.get_cmap("nipy_spectral")

    norm = mpl.colors.Normalize(vmin=rte_lims[ul][0], vmax=rte_lims[ul][1])  # <-- use rte_max here
    norms[ul] = norm
    # ---- HISTOGRAM COLORED BY VALUE ----
    # histogram → raw counts, not density
    counts, bin_edge = np.histogram(rte, bins="auto", density=False)
    bin_edges[ul] = bin_edge
    # convert counts → probability (sums to 1)
    prob = counts / counts.sum()
    probs[ul] = prob

    # bin centers (for coloring)
    bin_center = 0.5 * (bin_edge[:-1] + bin_edge[1:])
    bar_color = cmap(norm(bin_center))
    bin_centers[ul] = bin_center
    bar_colors[ul] = bar_color

In [ ]:
ps = PlotStyler(fontsize=20)
ps.apply_style()

fig, ax = plt.subplots(2, 3, figsize=(11, 6))
for i in range(6):
    ps.set_tick_padding(ax.flat[i], pad_x=6)
scs = []
for i, ul in enumerate(["U", "L"]):
    ax[i, 0].bar(bin_centers[ul], probs[ul], width=np.diff(bin_edges[ul]),
                color=bar_colors[ul], edgecolor='none')
    
    ax[i, 0].set_ylabel("Probability")
    ax[i, 0].set_xlim(*rte_lims[ul])
    # ax[0].set_yscale("log")
    sc = ax[i, 1].scatter(xs[ul], ys[ul], c=rtes[ul], s=.01, edgecolor="none",
                            cmap=cmap, norm=norms[ul])
    scs.append(sc)
    ax[i, 2].scatter(xs[ul], ys[ul], c=rtes[ul], s=.1, edgecolor="none",
                            cmap=cmap, norm=norms[ul])
    # ax[1].set_xlim(0.431, 0.445)
    # ax[1].set_ylim(0.31, 0.36)

ax[0, 1].set_xlim(0.42, 0.58)
ax[1, 1].set_xlim(0.42, 0.58)
ax[0, 1].set_ylim(0.02, 0.45)
ax[1, 1].set_ylim(-0.45, 0.02)
ax[-1, 0].set_xlabel(fr"$\mathrm{{RTE}}({finite_time})$")
ax[0, 2].set_xlim(0.49, 0.51)
ax[0, 2].set_ylim(0.223, 0.253)
ax[1, 2].set_xlim(0.49, 0.51)
ax[1, 2].set_ylim(-0.25, -0.22)
ax[0, 2].set_yticks(np.linspace(0.223, 0.253, 3))
# ax[0, 0].set_yticks([0, 0.002, 0.004, 0.006, 0.008, 0.01])


ax[0, 0].set_xticks([0, 0.5, 1, 1.5])
ax[1, 0].set_xticks([0, 0.5, 1, 1.5])
ax[0, 1].set_ylabel("$y$")
ax[1, 1].set_ylabel("$y$")
ax[1, 1].set_xlabel("$x$")
ax[1, 2].set_xlabel("$x$")
ax[0, 0].set_yscale("log")
ax[1, 0].set_yscale("log")
ax[0, 0].set_ylim(5e-6, 2e-2)
ax[1, 0].set_ylim(5e-6, 2e-2)

plt.subplots_adjust(left=0.082, bottom=0.095, right=1.08, top=0.985, wspace=0.37,
                    hspace=0.17)

xbox = 0.01
ybox = 0.912
bbox = {"facecolor": "w", "linewidth": 0.0, "pad": 1, "alpha": 0.75}
for i in range(6):
    ax.flat[i].text(xbox, ybox, f"({abc[i]})", transform=ax.flat[i].transAxes,
                    bbox=bbox)

for i in range(2):
    plt.colorbar(scs[i], ax=ax[i], label="RTE(200)", pad=0.01)
plt.savefig("figures/fig7.png", dpi=400)
plt.close()

# Fig. 8

In [ ]:
@njit
def mapping(u, parameters):
    x, y = u
    a, b, c, p, q = parameters
    m = p / q
    two_pi = 2 * np.pi
    y_new = y - b * np.sin(two_pi * x) - c * np.sin(two_pi * m * x)
    x_new = (x + a * (1 - y_new ** 2)) % q

    return np.array([x_new, y_new])

In [ ]:
ds = dds(mapping=mapping, system_dimension=2, number_of_parameters=5)

In [ ]:
c = 5e-3
p_vals = [1, 4, 5]
q_vals = [2, 5, 3]
num_ic = 150
total_time = 10000

In [ ]:
def get_trajectories(ds, total_time, parameters, x_range, y_range, num_ic):
    np.random.seed(1312)
    x = np.random.uniform(*x_range, num_ic)
    y = np.random.uniform(*y_range, num_ic)
    u = np.stack([x, y]).T
    return ds.trajectory(u, total_time, parameters)

In [ ]:
from matplotlib.gridspec import GridSpec
fontsize = 19
s = 0.06
ps = PlotStyler(fontsize=fontsize, markersize=0.1, markeredgewidth=0)
ps.apply_style()
fig = plt.figure(figsize=(10, 7))
gs = GridSpec(6, 4, figure=fig)

ax = []

parameters = [a, b, c, p_vals[0], q_vals[0]]
ax.append(fig.add_subplot(gs[0:2, 0]))
x_range = (0, q_vals[0])
y_range = (-0.5, 0.5)
traj = get_trajectories(ds, total_time, parameters, x_range, y_range, num_ic)
ax[0].scatter(traj[:, 0], traj[:, 1], c="k", s=s, edgecolor="none")
ax[0].set_xlim(*x_range)
ax[0].set_ylim(*y_range)
ax[0].set_ylabel("$y$")

ax.append(fig.add_subplot(gs[0, 1]))
x_range = (0.88, 1.1)
y_range = (0.0, 0.45)
xs = [x_range[0], x_range[1], x_range[1], x_range[0], x_range[0]]
ys = [y_range[0], y_range[0], y_range[1], y_range[1], y_range[0]]
ax[0].plot(xs, ys, "r--", linewidth=1)
traj = get_trajectories(ds, total_time, parameters, x_range, y_range, num_ic)
ax[1].scatter(traj[:, 0], traj[:, 1], c="k", s=s, edgecolor="none")
ax[1].set_xlim(*x_range)
ax[1].set_ylim(*y_range)
ax[1].set_yticks(np.linspace(*y_range, 3))

ax.append(fig.add_subplot(gs[1, 1]))
x_range = (0.88, 1.1)
y_range = (-0.45, 0.)
xs = [x_range[0], x_range[1], x_range[1], x_range[0], x_range[0]]
ys = [y_range[0], y_range[0], y_range[1], y_range[1], y_range[0]]
ax[0].plot(xs, ys, "b--", linewidth=1)
traj = get_trajectories(ds, total_time, parameters, x_range, y_range, num_ic)
ax[2].scatter(traj[:, 0], traj[:, 1], c="k", s=s, edgecolor="none")
ax[2].set_xlim(*x_range)
ax[2].set_ylim(*y_range)
ax[2].set_yticks(np.linspace(*y_range, 3))

ax.append(fig.add_subplot(gs[0, 2]))
x_range = (0.95, 1.05)
y_range = (0.19, 0.29)
xs = [x_range[0], x_range[1], x_range[1], x_range[0], x_range[0]]
ys = [y_range[0], y_range[0], y_range[1], y_range[1], y_range[0]]
ax[1].plot(xs, ys, "r--", linewidth=1)
traj = get_trajectories(ds, total_time, parameters, x_range, y_range, num_ic)
ax[3].scatter(traj[:, 0], traj[:, 1], c="k", s=s, edgecolor="none")
ax[3].set_xlim(*x_range)
ax[3].set_ylim(*y_range)
ax[3].set_yticks(np.linspace(*y_range, 3))

ax.append(fig.add_subplot(gs[1, 2]))
x_range = (0.95, 1.05)
y_range = (-0.29, -0.19)
xs = [x_range[0], x_range[1], x_range[1], x_range[0], x_range[0]]
ys = [y_range[0], y_range[0], y_range[1], y_range[1], y_range[0]]
ax[2].plot(xs, ys, "b--", linewidth=1)
traj = get_trajectories(ds, total_time, parameters, x_range, y_range, num_ic)
ax[4].scatter(traj[:, 0], traj[:, 1], c="k", s=s, edgecolor="none")
ax[4].set_xlim(*x_range)
ax[4].set_ylim(*y_range)
ax[4].set_yticks(np.linspace(*y_range, 3))

ax.append(fig.add_subplot(gs[0, 3]))
x_range = (0.995, 1.005)
y_range = (0.225, 0.24)
xs = [x_range[0], x_range[1], x_range[1], x_range[0], x_range[0]]
ys = [y_range[0], y_range[0], y_range[1], y_range[1], y_range[0]]
ax[3].plot(xs, ys, "r--", linewidth=1)
traj = get_trajectories(ds, total_time, parameters, x_range, y_range, num_ic)
ax[5].scatter(traj[:, 0], traj[:, 1], c="k", s=s, edgecolor="none")
ax[5].set_xlim(*x_range)
ax[5].set_ylim(*y_range)
ax[5].set_yticks(np.linspace(*y_range, 2))

ax.append(fig.add_subplot(gs[1, 3]))
x_range = (0.995, 1.005)
y_range = (-0.245, -0.23)
xs = [x_range[0], x_range[1], x_range[1], x_range[0], x_range[0]]
ys = [y_range[0], y_range[0], y_range[1], y_range[1], y_range[0]]
ax[4].plot(xs, ys, "b--", linewidth=1)
traj = get_trajectories(ds, total_time, parameters, x_range, y_range, num_ic)
ax[6].scatter(traj[:, 0], traj[:, 1], c="k", s=s, edgecolor="none")
ax[6].set_xlim(*x_range)
ax[6].set_ylim(*y_range)
ax[6].set_yticks(np.linspace(*y_range, 2))

#################################################
parameters = [a, b, c, p_vals[1], q_vals[1]]
ax.append(fig.add_subplot(gs[2:4, 0]))
x_range = (0, q_vals[1])
y_range = (-0.5, 0.5)
traj = get_trajectories(ds, total_time, parameters, x_range, y_range, num_ic)
ax[7].scatter(traj[:, 0], traj[:, 1], c="k", s=s, edgecolor="none")
ax[7].set_xlim(*x_range)
ax[7].set_ylim(*y_range)
ax[7].set_ylabel("$y$")

ax.append(fig.add_subplot(gs[2, 1]))
x_range = (2.4, 2.6)
y_range = (0, 0.45)
xs = [x_range[0], x_range[1], x_range[1], x_range[0], x_range[0]]
ys = [y_range[0], y_range[0], y_range[1], y_range[1], y_range[0]]
ax[7].plot(xs, ys, "r--", linewidth=1)
traj = get_trajectories(ds, total_time, parameters, x_range, y_range, num_ic)
ax[8].scatter(traj[:, 0], traj[:, 1], c="k", s=s, edgecolor="none")
ax[8].set_xlim(*x_range)
ax[8].set_ylim(*y_range)
ax[8].set_yticks(np.linspace(*y_range, 3))

ax.append(fig.add_subplot(gs[3, 1]))
x_range = (2.4, 2.6)
y_range = (-0.45, 0.0)
xs = [x_range[0], x_range[1], x_range[1], x_range[0], x_range[0]]
ys = [y_range[0], y_range[0], y_range[1], y_range[1], y_range[0]]
ax[7].plot(xs, ys, "b--", linewidth=1)
traj = get_trajectories(ds, total_time, parameters, x_range, y_range, num_ic)
ax[9].scatter(traj[:, 0], traj[:, 1], c="k", s=s, edgecolor="none")
ax[9].set_xlim(*x_range)
ax[9].set_ylim(*y_range)
ax[9].set_yticks(np.linspace(*y_range, 3))

ax.append(fig.add_subplot(gs[2, 2]))
x_range = (2.45, 2.55)
y_range = (0.2, 0.28)
xs = [x_range[0], x_range[1], x_range[1], x_range[0], x_range[0]]
ys = [y_range[0], y_range[0], y_range[1], y_range[1], y_range[0]]
ax[8].plot(xs, ys, "r--", linewidth=1)
traj = get_trajectories(ds, total_time, parameters, x_range, y_range, num_ic)
ax[10].scatter(traj[:, 0], traj[:, 1], c="k", s=s, edgecolor="none")
ax[10].set_xlim(*x_range)
ax[10].set_ylim(*y_range)
ax[10].set_yticks(np.linspace(*y_range, 3))

ax.append(fig.add_subplot(gs[3, 2]))
x_range = (2.45, 2.55)
y_range = (-0.27, -0.19)
xs = [x_range[0], x_range[1], x_range[1], x_range[0], x_range[0]]
ys = [y_range[0], y_range[0], y_range[1], y_range[1], y_range[0]]
ax[9].plot(xs, ys, "b--", linewidth=1)
traj = get_trajectories(ds, total_time, parameters, x_range, y_range, num_ic)
ax[11].scatter(traj[:, 0], traj[:, 1], c="k", s=s, edgecolor="none")
ax[11].set_xlim(*x_range)
ax[11].set_ylim(*y_range)
ax[11].set_yticks(np.linspace(*y_range, 3))

ax.append(fig.add_subplot(gs[2, 3]))
x_range = (2.495, 2.505)
y_range = (0.23, 0.25)
xs = [x_range[0], x_range[1], x_range[1], x_range[0], x_range[0]]
ys = [y_range[0], y_range[0], y_range[1], y_range[1], y_range[0]]
ax[10].plot(xs, ys, "r--", linewidth=1)
traj = get_trajectories(ds, total_time, parameters, x_range, y_range, num_ic)
ax[12].scatter(traj[:, 0], traj[:, 1], c="k", s=s, edgecolor="none")
ax[12].set_xlim(*x_range)
ax[12].set_ylim(*y_range)
ax[12].set_yticks(np.linspace(*y_range, 3))

ax.append(fig.add_subplot(gs[3, 3]))
x_range = (2.495, 2.505)
y_range = (-0.24, -0.22)
xs = [x_range[0], x_range[1], x_range[1], x_range[0], x_range[0]]
ys = [y_range[0], y_range[0], y_range[1], y_range[1], y_range[0]]
ax[11].plot(xs, ys, "b--", linewidth=1)
traj = get_trajectories(ds, total_time, parameters, x_range, y_range, num_ic)
ax[13].scatter(traj[:, 0], traj[:, 1], c="k", s=s, edgecolor="none")
ax[13].set_xlim(*x_range)
ax[13].set_ylim(*y_range)
ax[13].set_yticks(np.linspace(*y_range, 3))

################################################################
parameters = [a, b, c, p_vals[2], q_vals[2]]
ax.append(fig.add_subplot(gs[4:6, 0]))
x_range = (0, q_vals[2])
y_range = (-0.5, 0.5)
traj = get_trajectories(ds, total_time, parameters, x_range, y_range, num_ic)
ax[14].scatter(traj[:, 0], traj[:, 1], c="k", s=s, edgecolor="none")
ax[14].set_xlim(*x_range)
ax[14].set_ylim(*y_range)
ax[14].set_ylabel("$y$")
ax[14].set_xlabel("$x$")
ax[14].set_xticks([0, 1, 2, 3])

ax.append(fig.add_subplot(gs[4, 1]))
x_range = (1.35, 1.65)
y_range = (0, 0.45)
xs = [x_range[0], x_range[1], x_range[1], x_range[0], x_range[0]]
ys = [y_range[0], y_range[0], y_range[1], y_range[1], y_range[0]]
ax[14].plot(xs, ys, "r--", linewidth=1)
traj = get_trajectories(ds, total_time, parameters, x_range, y_range, num_ic)
ax[15].scatter(traj[:, 0], traj[:, 1], c="k", s=s, edgecolor="none")
ax[15].set_xlim(*x_range)
ax[15].set_ylim(*y_range)
ax[15].set_yticks(np.linspace(*y_range, 3))

ax.append(fig.add_subplot(gs[5, 1]))
x_range = (1.35, 1.65)
y_range = (-0.45, 0)
xs = [x_range[0], x_range[1], x_range[1], x_range[0], x_range[0]]
ys = [y_range[0], y_range[0], y_range[1], y_range[1], y_range[0]]
ax[14].plot(xs, ys, "b--", linewidth=1)
traj = get_trajectories(ds, total_time, parameters, x_range, y_range, num_ic)
ax[16].scatter(traj[:, 0], traj[:, 1], c="k", s=s, edgecolor="none")
ax[16].set_xlim(*x_range)
ax[16].set_ylim(*y_range)
ax[16].set_xlabel("$x$")
ax[16].set_yticks(np.linspace(*y_range, 3))


ax.append(fig.add_subplot(gs[4, 2]))
x_range = (1.45, 1.55)
y_range = (0.2, 0.28)
xs = [x_range[0], x_range[1], x_range[1], x_range[0], x_range[0]]
ys = [y_range[0], y_range[0], y_range[1], y_range[1], y_range[0]]
ax[15].plot(xs, ys, "r--", linewidth=1)
traj = get_trajectories(ds, total_time, parameters, x_range, y_range, num_ic)
ax[17].scatter(traj[:, 0], traj[:, 1], c="k", s=s, edgecolor="none")
ax[17].set_xlim(*x_range)
ax[17].set_ylim(*y_range)
ax[17].set_yticks(np.linspace(*y_range, 3))

ax.append(fig.add_subplot(gs[5, 2]))
x_range = (1.45, 1.55)
y_range = (-0.27, -0.19)
xs = [x_range[0], x_range[1], x_range[1], x_range[0], x_range[0]]
ys = [y_range[0], y_range[0], y_range[1], y_range[1], y_range[0]]
ax[16].plot(xs, ys, "b--", linewidth=1)
traj = get_trajectories(ds, total_time, parameters, x_range, y_range, num_ic)
ax[18].scatter(traj[:, 0], traj[:, 1], c="k", s=s, edgecolor="none")
ax[18].set_xlim(*x_range)
ax[18].set_ylim(*y_range)
ax[18].set_xlabel("$x$")
ax[18].set_yticks(np.linspace(*y_range, 3))


ax.append(fig.add_subplot(gs[4, 3]))
x_range = (1.495, 1.505)
y_range = (0.232, 0.255)
xs = [x_range[0], x_range[1], x_range[1], x_range[0], x_range[0]]
ys = [y_range[0], y_range[0], y_range[1], y_range[1], y_range[0]]
ax[17].plot(xs, ys, "r--", linewidth=1)
traj = get_trajectories(ds, total_time, parameters, x_range, y_range, num_ic)
ax[19].scatter(traj[:, 0], traj[:, 1], c="k", s=s, edgecolor="none")
ax[19].set_xlim(*x_range)
ax[19].set_ylim(*y_range)
ax[19].set_xticks([1.495, 1.5 ,1.505])
ax[19].set_yticks(np.linspace(*y_range, 2))

ax.append(fig.add_subplot(gs[5, 3]))
x_range = (1.495, 1.505)
y_range = (-0.24, -0.22)
xs = [x_range[0], x_range[1], x_range[1], x_range[0], x_range[0]]
ys = [y_range[0], y_range[0], y_range[1], y_range[1], y_range[0]]
ax[18].plot(xs, ys, "b--", linewidth=1)
traj = get_trajectories(ds, total_time, parameters, x_range, y_range, num_ic)
ax[20].scatter(traj[:, 0], traj[:, 1], c="k", s=s, edgecolor="none")
ax[20].set_xlim(*x_range)
ax[20].set_ylim(*y_range)
ax[20].set_xlabel("$x$")
ax[20].set_xticks([1.495, 1.5 ,1.505])
ax[20].set_yticks(np.linspace(*y_range, 3))

for i in range(len(ax)):
    ps.set_tick_padding(ax[i], pad_x=7)

xbox = 0.015
ybox = 0.89
bbox = {"facecolor": "w", "linewidth": 0.0, "pad": 1, "alpha": 0.75}
ax[0].text(xbox, ybox, f"(a)", transform=ax[0].transAxes, bbox=bbox)
ax[7].text(xbox, ybox, f"(h)", transform=ax[7].transAxes, bbox=bbox)
ax[14].text(xbox, ybox, f"(o)", transform=ax[14].transAxes, bbox=bbox)
idx_vals = np.delete(np.arange(0, 21), [0, 7, 14])
idx_vals = [1, 3, 5, 2, 4, 6,
            8, 10, 12, 9, 11, 13,
            15, 17, 19, 16, 18, 20]
abc = ["b", "c", "d", "e", "f", "g",
       "i", "j", "k", "l", "m", "n",
       "p", "q", "r", "s", "t", "u"]
xbox2 = 0.0145
ybox2 = 0.792
for i, idx in enumerate(idx_vals):
    ax[idx].text(xbox2, ybox2, f"({abc[i]})", transform=ax[idx].transAxes, bbox=bbox, fontsize=14)


plt.subplots_adjust(left=0.084, bottom=0.08, right=0.975, top=0.984, hspace=0.45, wspace=0.41)
plt.savefig("figures/fig8.png", dpi=400)
plt.close()